# Database Subclass Breakdown

In [26]:
import pandas as pd
import sqlite3
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

DB_PATH = "../../CCSMLDatabase.db"
TABLE = "master_clean"

conn = sqlite3.connect(DB_PATH)
df = pd.read_sql_query(
    f"SELECT * FROM master_clean",
    conn,
)
conn.close()

# Global style settings
plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"], # Standard crisp fonts
    "axes.linewidth": 1.5,               # Thicker axis lines
    "axes.spines.top": False,            # Remove top spine
    "axes.spines.right": False,          # Remove right spine
    "xtick.major.width": 1.5,            # Match tick thickness to axis
    "ytick.major.width": 1.5,
    "xtick.direction": "out",            # Ticks point outside
    "ytick.direction": "out",
    "font.size": 10,
    "axes.labelsize": 11,
    "axes.labelweight": "normal",
})

# Color palette for consistency
COLORS = {
    'primary': '#2c7bb6',
    'secondary': '#d7191c',
    'tertiary': '#fdae61',
    'quaternary': '#abd9e9',
    'positive': '#2ca02c',
    'negative': '#d62728',
}

n = df.shape[0]
df.columns

Index(['id', 'tag', 'name', 'pubchemId', 'adduct', 'mass', 'z', 'ccs', 'smi',
       'inchikey', 'superclass', 'class', 'subclass'],
      dtype='object')

In [27]:
# Subclass counts and percent of total dataset
# Only real, non-predicted subclass labels are counted: null subclasses and any
# label containing "(predicted)" are excluded from subclass_counts entirely.
is_predicted = df['subclass'].str.contains(r'\(predicted\)', case=False, na=False)
n_null = df['subclass'].isna().sum()
n_predicted = is_predicted.sum()

subclass_counts = df.loc[df['subclass'].notna() & ~is_predicted, 'subclass'].value_counts()

print(f"Total datapoints: {n:,}")
print(f"Unique (non-predicted) subclasses: {subclass_counts.shape[0]}")
print(f"Null subclass datapoints: {n_null:,} ({n_null / n * 100:.1f}% of total)")
print(f"Predicted subclass datapoints (excluded): {n_predicted:,} ({n_predicted / n * 100:.1f}% of total)")

Total datapoints: 66,153
Unique (non-predicted) subclasses: 435
Null subclass datapoints: 9,724 (14.7% of total)
Predicted subclass datapoints (excluded): 6,360 (9.6% of total)


In [28]:
# Build top-50 subclass summary (percent of *total* dataset). "Other" covers only
# remaining real, non-predicted subclasses beyond the top 50 — it excludes null and
# predicted subclass datapoints entirely (those are reported separately above).
top50 = subclass_counts.head(50)

summary = top50.reset_index()
summary.columns = ['Subclass', 'Count']
summary['Percent of Dataset'] = (summary['Count'] / n * 100).round(2)
summary.insert(0, 'Rank', range(1, len(summary) + 1))

top50_total_pct = summary['Percent of Dataset'].sum()
other_count = subclass_counts.sum() - summary['Count'].sum()
other_row = pd.DataFrame([{
    'Rank': '-',
    'Subclass': 'Other (remaining subclasses)',
    'Count': other_count,
    'Percent of Dataset': round(other_count / n * 100, 2),
}])
summary = pd.concat([summary, other_row], ignore_index=True)
summary['Cumulative Percent'] = summary['Percent of Dataset'].cumsum().round(2)

print(f"The top 50 subclasses account for {top50_total_pct:.1f}% of all {n:,} datapoints in the dataset.")
summary

The top 50 subclasses account for 57.9% of all 66,153 datapoints in the dataset.


,Rank,Subclass,Count,Percent of Dataset,Cumulative Percent
0,1,"Amino acids, peptides, and analogues",5979,9.04,9.04
1,2,Benzoic acids and derivatives,3911,5.91,14.95
2,3,Anilides,2514,3.80,18.75
3,4,Benzenesulfonamides,1824,2.76,21.51
4,5,Biphenyls and derivatives,1568,2.37,23.88
5,6,Piperazines,1384,2.09,25.97
6,7,Carbonyl compounds,1288,1.95,27.92
7,8,Triazoles,1239,1.87,29.79
8,9,Pyrazoles,1196,1.81,31.60
9,10,Benzodiazines,1009,1.53,33.13


In [29]:
# Top-50 table with predicted subclasses merged into their ground-truth counterpart
# e.g. "Amino acids, peptides, and analogues (predicted)" is counted together with
# "Amino acids, peptides, and analogues". "Other" covers only remaining merged
# subclasses beyond the top 50 — it excludes null subclass datapoints entirely.
merged_subclass = df['subclass'].str.replace(r'\s*\(predicted\)', '', case=False, regex=True)
merged_counts = merged_subclass.dropna().value_counts()

merged_top50 = merged_counts.head(50)

merged_summary = merged_top50.reset_index()
merged_summary.columns = ['Subclass', 'Count']
merged_summary['Percent of Dataset'] = (merged_summary['Count'] / n * 100).round(2)
merged_summary.insert(0, 'Rank', range(1, len(merged_summary) + 1))

merged_top50_total_pct = merged_summary['Percent of Dataset'].sum()
merged_other_count = merged_counts.sum() - merged_summary['Count'].sum()
merged_other_row = pd.DataFrame([{
    'Rank': '-',
    'Subclass': 'Other (remaining subclasses)',
    'Count': merged_other_count,
    'Percent of Dataset': round(merged_other_count / n * 100, 2),
}])
merged_summary = pd.concat([merged_summary, merged_other_row], ignore_index=True)
merged_summary['Cumulative Percent'] = merged_summary['Percent of Dataset'].cumsum().round(2)

print(f"With predicted subclasses merged into their ground-truth label, the top 50 "
      f"subclasses account for {merged_top50_total_pct:.1f}% of all {n:,} datapoints in the dataset.")
merged_summary

With predicted subclasses merged into their ground-truth label, the top 50 subclasses account for 66.7% of all 66,153 datapoints in the dataset.


,Rank,Subclass,Count,Percent of Dataset,Cumulative Percent
0,1,"Amino acids, peptides, and analogues",7237,10.94,10.94
1,2,Benzoic acids and derivatives,4301,6.50,17.44
2,3,Anilides,2788,4.21,21.65
3,4,Benzenesulfonamides,2126,3.21,24.86
4,5,Biphenyls and derivatives,1803,2.73,27.59
5,6,Piperazines,1603,2.42,30.01
6,7,Triazoles,1579,2.39,32.40
7,8,Carbonyl compounds,1416,2.14,34.54
8,9,Pyrazoles,1385,2.09,36.63
9,10,Aryl thioethers,1207,1.82,38.45
